# Aufgabe 0: Wie funktioniert ein LLM-API-Call?

Voraussetzung: Setup aus README abgeschlossen.

In [21]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b nicht gefunden: {models}"
print("OK")

OK


## Teil 1: Ein LLM-Call

`ollama.chat()` nimmt eine Liste von Messages und gibt ein `ChatResponse`-Objekt zurueck.

In [22]:
import json

response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Was ist 2 + 2?"}],
)

print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "model": "qwen3.5:4b",
  "created_at": "2026-04-14T08:46:09.655363117Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 14017639864,
  "load_duration": 6150267970,
  "prompt_eval_count": 18,
  "prompt_eval_duration": 43254901,
  "eval_count": 429,
  "eval_duration": 7695647059,
  "message": {
    "role": "assistant",
    "content": "2 + 2 ist **4**.",
    "thinking": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: \"Was ist 2 + 2?\" (German for \"What is 2 + 2?\")\n    *   Task: Answer the mathematical question.\n    *   Language: German (matching the input).\n\n2.  **Determine the Answer:**\n    *   2 + 2 = 4.\n\n3.  **Formulate the Output:**\n    *   Direct and clear: \"2 + 2 ist 4.\"\n    *   Optional: Add a bit more context or keep it simple? Usually, for such a simple query, a direct answer is best.\n    *   Draft: 2 + 2 = 4.\n    *   Draft (more natural): 2 plus 2 ergibt 4.\n    *   Draft (concise): 4.\n\n    Let's go with a clear sentence. \"

## Teil 2: Statelessness

Die Chat API ist stateless. Das Modell sieht bei jedem Call nur die `messages` die wir in diesem Call mitschicken.

In [23]:
# Ohne Conversation History: zwei getrennte Calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Wie heisse ich?"}])
print("Call 2:", r2.message.content)

Call 1: Hallo Max! Schön, dich kennenzulernen. 

Wie kann ich dir heute helfen?
Call 2: Ich kenne Ihren Namen leider nicht, da ich keine persönlichen Informationen über Sie gespeichert habe. Falls Sie mir Ihren Namen nennen möchten, helfe ich Ihnen gerne weiter!


In [24]:
# Mit Conversation History: gleiche zwei Calls, aber History mitschicken
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "Ich heisse Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "Wie heisse ich?"},
])
print("Call 2:", r2.message.content)

Call 1: Hallo Max! Es freut mich sehr, dich kennenzulernen. Wie kann ich dir heute helfen?
Call 2: Dein Name ist ja Max! 😊 Das ist schön, dass du es mir geschrieben hast. Wie kann ich dir heute helfen?


## Teil 3: Tool Calling

Mit dem Parameter `tools=` uebergeben wir eine Liste von **Tool Definitions**. Jede Definition hat einen Namen, eine Description und ein JSON Schema fuer die Parameter. Das Modell kann dann einen Tool Call mit `function.name` und `function.arguments` zurueckgeben — in dem Fall ist `content` typischerweise leer. Es fuehrt die Funktion nicht selbst aus.

In [25]:
# Tool Definition: Name, Description, Parameter-Schema
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

In [28]:
# Gleiche Frage — einmal ohne Tools, einmal mit
frage = [{"role": "user", "content": "Berechne 17 * 23"}]

r_ohne = ollama.chat(model="qwen3.5:4b", messages=frage)
print("Ohne Tools")
print("  content:   ", r_ohne.message.content)
print("  tool_calls:", r_ohne.message.tool_calls)

r_mit = ollama.chat(model="qwen3.5:4b", messages=frage, tools=calculator_tool)
print("\nMit Tools")
print("  content:   ", r_mit.message.content or "(leer)")
print("  tool_calls:", r_mit.message.tool_calls)

Ohne Tools
  content:    Die Rechnung ergibt:

17 · 23 = 391

Hier ist die Aufschlüsselung:
*   17 · 20 = 340
*   17 · 3 = 51
*   340 + 51 = 391
  tool_calls: None

Mit Tools
  content:    (leer)
  tool_calls: [ToolCall(function=Function(name='calculator', arguments={'expression': '17 * 23'}))]


## Teil 4: Tool-Use Loop

Das Modell hat einen Tool Call zurueckgegeben aber nichts ausgefuehrt. Jetzt muessen **wir**:
1. Tool ausfuehren
2. Ergebnis als `{"role": "tool"}` Message zurueck schicken
3. Modell nochmal aufrufen → finale Antwort

In [27]:
# Schritt 1: User fragt
messages = [{"role": "user", "content": "Berechne 17 * 23"}]

# Schritt 2: LLM gibt Tool Call zurueck
r1 = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
tc = r1.message.tool_calls[0]
print(f"Tool Call: {tc.function.name}({tc.function.arguments})")

# Schritt 3: WIR fuehren das Tool aus
result = str(eval(tc.function.arguments["expression"]))
print(f"Ergebnis:  {result}")

# Schritt 4: Ergebnis zurueck ans Modell
messages.append(r1.message.model_dump())
messages.append({"role": "tool", "content": result})

r2 = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print(f"Antwort:   {r2.message.content}")

Tool Call: calculator({'expression': '17 * 23'})
Ergebnis:  391
Antwort:   17 * 23 = 391


## Zusammenfassung

1. `ollama.chat()` nimmt eine `messages`-Liste und gibt ein `ChatResponse`-Objekt zurueck.
2. Die API ist stateless — das Modell sieht nur die Messages die wir in diesem Call mitschicken.
3. Mit `tools=` kann das Modell Tool Calls zurueckgeben. `content` ist dann typischerweise leer. Die Ausfuehrung liegt bei uns.
4. Der Tool-Use Loop (User → LLM → Tool Call → Ausfuehrung → Ergebnis zurueck → LLM → Antwort) ist das was ein Agent automatisiert.

→ **Aufgabe 1**: diesen Loop implementieren.